In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import gdown 

url = "https://drive.google.com/file/d/1A40POUq1g1EqSlE76zqqyHo4KqzLVtub/view?usp=sharing"
output_filename = 'LogisticRegressionModel.pkl'

# Use fuzzy=True so gdown strips away the Google HTML warning pages
file_path = gdown.download(url, output_filename, quiet=False, fuzzy=True)

# Read the file now that it is properly saved on disk
LRModel = pd.read_pickle(file_path)

In [ ]:


import gdown 

url = "https://drive.google.com/file/d/1BhsF97-CgeZj5UDT5WozXC564KtaQ2WK/view?usp=sharing"
output_filename = 'LogisticVectorizer.pkl'

# Use fuzzy=True so gdown strips away the Google HTML warning pages
file_path = gdown.download(url, output_filename, quiet=False, fuzzy=True)

# Read the file now that it is properly saved on disk
LV = pd.read_pickle(file_path)

In [ ]:
comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "You deserve to get hurt",
    "He is a Muslim teacher",
    "I hate Muslims",
    "Everybody secretly hates you",
    "He is black",
    "He is an annoying black dude",
    "She is a woman",
    "She is a nice woman",
    "She is not bad"
]

# Transform text
comments_tfidf = LV.transform(comments)

# Predict probabilities
pred_probs = LRModel.predict_proba(comments_tfidf)[:, 1]

# Convert to labels
pred_labels = (pred_probs >= 0.5).astype(int)

# Print results
for comment, prob, label in zip(comments, pred_probs, pred_labels):
    print(f"Comment: {comment}")
    print(f"Toxic Probability: {prob:.4f}")
    print(f"Prediction: {'Toxic' if label == 1 else 'Non-Toxic'}")
    print("-" * 60)

In [ ]:
from tensorflow.keras.models import load_model

SingleLSTM_model = load_model('/kaggle/input/models/debangshudey2004/singlelstm-model/keras/default/1/SingleLSTM_Model.keras')

In [ ]:
import gdown
MAX_SEQUENCE_LENGTH = 300

url = "https://drive.google.com/file/d/15PWNsDqb-QFg34YQ90GUGnO9gi1YQEgs/view?usp=sharing"
output_filename = 'tokenizer.pkl'

# Use fuzzy=True so gdown strips away the Google HTML warning pages
file_path = gdown.download(url, output_filename, quiet=False, fuzzy=True)

# Read the file now that it is properly saved on disk
tokenizer = pd.read_pickle(file_path)



from tensorflow.keras.preprocessing.sequence import pad_sequences

def padding_text(texts):
    return pad_sequences(
        texts,
        maxlen=MAX_SEQUENCE_LENGTH
    )

In [ ]:
comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "You deserve to get hurt",
    "He is a Muslim teacher",
    "I hate Muslims",
    "Everybody secretly hates you",
    "He is black",
    "He is an annoying black dude",
    "She is a woman",
    "She is a nice woman",
    "She is not bad"
]

for sentence in comments:

    seq = tokenizer.texts_to_sequences([sentence])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH
    )

    pred = SingleLSTM_model.predict(padded, verbose=0)

    print(sentence)
    print(pred)
    print("-"*50)

In [ ]:
BidirectionalLSTM_model = load_model('/kaggle/input/models/debangshudey2004/bidirectionallstm-model/keras/default/1/BidirectionalLSTM_Model.keras')

In [ ]:
comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "You deserve to get hurt",
    "He is a Muslim teacher",
    "I hate Muslims",
    "Everybody secretly hates you",
    "He is black",
    "He is an annoying black dude",
    "She is a woman",
    "She is a nice woman",
    "She is not bad"
]


for sentence in comments:

    seq = tokenizer.texts_to_sequences([sentence])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH
    )

    pred = BidirectionalLSTM_model.predict(padded, verbose=0)

    print(sentence)
    print(pred)
    print("-"*50)

In [ ]:
WSBLSTM_model = load_model('/kaggle/input/models/debangshudey2004/weightedsamplesbidirectionallstm-model/keras/default/1/WeightedSamplesBidirectionalLSTM_Model.keras')

In [ ]:
comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "You deserve to get hurt",
    "He is a Muslim teacher",
    "I hate Muslims",
    "Everybody secretly hates you",
    "He is black",
    "He is an annoying black dude",
    "She is a woman",
    "She is a nice woman",
    "She is not bad"
]

for sentence in comments:

    seq = tokenizer.texts_to_sequences([sentence])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH
    )

    pred = WSBLSTM_model.predict(padded, verbose=0)

    print(sentence)
    print(pred)
    print("-"*50)

In [ ]:
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K
from tensorflow.keras import initializers, regularizers, constraints

import tensorflow as tf
class Attention(Layer):
    def __init__(self, step_dim,
                 W_regularizer=None, b_regularizer=None,
                 W_constraint=None, b_constraint=None,
                 bias=True, **kwargs):
        self.supports_masking = True
        self.init = initializers.get('glorot_uniform')

        self.W_regularizer = regularizers.get(W_regularizer)
        self.b_regularizer = regularizers.get(b_regularizer)

        self.W_constraint = constraints.get(W_constraint)
        self.b_constraint = constraints.get(b_constraint)

        self.bias = bias
        self.step_dim = step_dim
        self.features_dim = 0
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 3

        self.W = self.add_weight((input_shape[-1],),
                                 initializer=self.init,
                                 name='{}_W'.format(self.name),
                                 regularizer=self.W_regularizer,
                                 constraint=self.W_constraint)
        self.features_dim = input_shape[-1]

        if self.bias:
            self.b = self.add_weight((input_shape[1],),
                                     initializer='zero',
                                     name='{}_b'.format(self.name),
                                     regularizer=self.b_regularizer,
                                     constraint=self.b_constraint)
        else:
            self.b = None

        self.built = True

    def compute_mask(self, input, input_mask=None):
        return None

    def call(self, x, mask=None):
        features_dim = self.features_dim
        step_dim = self.step_dim
    
        # Replace K.reshape + K.dot with TensorFlow ops
        eij = tf.reshape(
            tf.matmul(
                tf.reshape(x, (-1, features_dim)),
                tf.reshape(self.W, (features_dim, 1))
            ),
            (-1, step_dim)
        )
    
        if self.bias:
            eij += self.b
    
        eij = tf.tanh(eij)
    
        a = tf.exp(eij)
    
        if mask is not None:
            a *= tf.cast(mask, tf.float32)
    
        a /= tf.cast(
            tf.reduce_sum(a, axis=1, keepdims=True) + K.epsilon(),
            tf.float32
        )
    
        a = tf.expand_dims(a, axis=-1)
    
        weighted_input = x * a
    
        return tf.reduce_sum(weighted_input, axis=1)

    def compute_output_shape(self, input_shape):
        return input_shape[0],  self.features_dim

In [ ]:
GRU_CONV1D_model = load_model(
    '/kaggle/input/models/debangshudey2004/gru-conv1d-model/keras/default/1/GRU_CONV1D_Model.keras',
    custom_objects={'Attention': Attention}
)

In [ ]:
comments = [
    "I hope you have a wonderful day",
    "You are being annoying",
    "You are a complete idiot",
    "I will destroy your life",
    "You are a fucking loser",
    "He is a Muslim man",
    "Great job ruining everything"
]

comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "Nobody wants you here",
    "You deserve to get hurt",
    "Wow, brilliant move genius",
    "He is a Muslim teacher",
    "I hate Muslims",
    "You murdered that presentation",
    "Cry more kid",
    "Everybody secretly hates you"
]

for sentence in comments:

    seq = tokenizer.texts_to_sequences([sentence])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH
    )

    pred = GRU_CONV1D_model.predict(padded, verbose=0)

    print(sentence)
    print(pred)
    print("-"*50)

In [ ]:
DenseGRU_CONV1D_model = load_model(
    '/kaggle/input/models/debangshudey2004/densegru-conv1d-model/keras/default/1/DenseGRU_CONV1D_Model.keras')

In [ ]:
comments = [
    "I hope you have a wonderful day",
    "You are being annoying",
    "You are a complete idiot",
    "I will destroy your life",
    "You are a fucking loser",
    "He is a Muslim man",
    "Great job ruining everything"
]

comments = [
    "I appreciate your help",
    "That was a very stupid thing to say",
    "Nobody wants you here",
    "You deserve to get hurt",
    "Wow, brilliant move genius",
    "He is a Muslim teacher",
    "I hate Muslims",
    "You murdered that presentation",
    "Cry more kid",
    "Everybody secretly hates you"
]

for sentence in comments:

    seq = tokenizer.texts_to_sequences([sentence])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH
    )

    pred = DenseGRU_CONV1D_model.predict(padded, verbose=0)

    print(sentence)
    print(pred)
    print("-"*50)